# Estimasi Curah Hujan Bulanan di Pulau Jawa
### Perbandingan Inverse Distance Weighted (IDW) & Multilayer Perceptron (MLP)

**Ringkasan proyek:**
Notebook ini merapikan ulang alur kerja skripsi menjadi satu pipeline yang runtut, dari data mentah hingga hasil akhir, lengkap dengan angka evaluasi final sesuai laporan skripsi. Tujuannya adalah mengestimasi distribusi curah hujan bulanan di wilayah Pulau Jawa (2001–2014) menggunakan data satelit PERSIANN yang dikoreksi bias terhadap data referensi MSWEP, lalu diinterpolasi secara spasial dengan tiga model yang dibandingkan: **IDW1**, **IDW2**, dan **MLP**.

**Definisi tiga model (sesuai metodologi skripsi):**
- **IDW1**: data PERSIANN dikoreksi bias terlebih dahulu → baru diinterpolasi secara spasial
- **IDW2**: data PERSIANN mentah diinterpolasi dulu → hasil interpolasinya baru dikoreksi bias terhadap MSWEP
- **MLP**: dilatih sekali menggunakan data yang sudah dikoreksi bias (input sama seperti IDW1)

**Alur pipeline:**
1. Preprocessing data (konversi format, pemisahan data bulanan)
2. Regridding (penyamaan resolusi spasial antar sumber data)
3. Bias correction data PERSIANN terhadap MSWEP (untuk input IDW1 & MLP)
4. IDW1 — interpolasi data yang sudah terkoreksi, dengan optimasi parameter power & radius
5. IDW2 — interpolasi data mentah, lalu koreksi bias pada hasil interpolasi
6. MLP — pemodelan dengan neural network 3 hidden layer
7. Evaluasi & hasil akhir — MAE bulanan/musiman/tahunan, POD/FAR/CSI untuk curah hujan ekstrem

> **Catatan:** seluruh path folder (`/content/drive/MyDrive/...`) mengikuti struktur Google Drive milik penulis saat pengerjaan skripsi. Sesuaikan path tersebut dengan lokasi datamu sendiri sebelum menjalankan ulang notebook ini.

## 1. Setup
Mount Google Drive dan import pustaka utama yang digunakan di sepanjang notebook.

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
#os.makedirs(drive_folder, exist_ok=True)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path


## 2. Preprocessing Data
Data curah hujan satelit (PERSIANN) dan data referensi (MSWEP) tersedia dalam format yang berbeda-beda (CSV dan NetCDF). Bagian ini mengonversi antar format dan memisahkan data time series menjadi file bulanan agar mudah diproses pada tahap berikutnya.

### 2.1 CSV → NetCDF

In [ ]:
# CSV TO NC
import xarray as xr
import pandas as pd
import os
import numpy as np
from datetime import datetime

# Path folder input dan output
input_folder = "/content/drive/MyDrive/PERSIANN_Validatioan_car2"
output_folder = "/content/drive/MyDrive/PERSIANN_Validation_cara2_NC"
os.makedirs(output_folder, exist_ok=True)

def process_csv_file(file_path):
    # Read CSV file
    df = pd.read_csv(file_path)

    # Convert datetime column to datetime object
    if 'datetime' in df.columns:
        df['datetime'] = pd.to_datetime(df['datetime'])
    else:
        raise ValueError("Datetime column not found in CSV file")

    # Group by datetime to reconstruct time series
    grouped = df.groupby('datetime')

    # Initialize lists to store data arrays
    time_list = []
    data_arrays = {}

    # Get variable names (exclude coordinate columns)
    vars_to_include = [col for col in df.columns if col not in ['datetime', 'lat', 'lon', 'latitude', 'longitude']]

    for time, group in grouped:
        time_list.append(time)

        # For each variable, create or append to data array
        for var in vars_to_include:
            # Reshape data if needed (assuming 2D spatial data)
            # This part might need adjustment based on your specific data structure
            if 'lat' in group.columns and 'lon' in group.columns:
                # Create 2D grid
                lats = group['lat'].unique()
                lons = group['lon'].unique()
                lats.sort()
                lons.sort()

                # Create empty array
                arr = np.empty((len(lats), len(lons)))
                arr[:] = np.nan

                # Fill array
                for _, row in group.iterrows():
                    lat_idx = np.where(lats == row['lat'])[0][0]
                    lon_idx = np.where(lons == row['lon'])[0][0]
                    arr[lat_idx, lon_idx] = row[var]

                if var not in data_arrays:
                    data_arrays[var] = []
                data_arrays[var].append(arr)

    # Create xarray Dataset
    ds = xr.Dataset()

    # Add time coordinate
    ds.coords['time'] = ('time', time_list)

    # Add spatial coordinates if available
    if 'lat' in df.columns and 'lon' in df.columns:
        lats = df['lat'].unique()
        lons = df['lon'].unique()
        lats.sort()
        lons.sort()
        ds.coords['lat'] = ('lat', lats)
        ds.coords['lon'] = ('lon', lons)

    # Add variables to dataset
    for var in data_arrays:
        # Determine dimensions based on available coordinates
        if 'lat' in ds.coords and 'lon' in ds.coords:
            dims = ('time', 'lat', 'lon')
        else:
            dims = ('time',)

        # Convert list of arrays to xarray DataArray
        arr = np.array(data_arrays[var])
        da = xr.DataArray(
            data=arr,
            dims=dims,
            coords={dim: ds.coords[dim] for dim in dims},
            name=var
        )
        ds[var] = da

    # Add attributes if needed
    ds.attrs['created_from'] = os.path.basename(file_path)
    ds.attrs['creation_date'] = str(datetime.now())

    return ds

# Loop semua file dalam folder
for file_name in os.listdir(input_folder):
    if file_name.endswith(".csv"):
        file_path = os.path.join(input_folder, file_name)
        try:
            ds = process_csv_file(file_path)
            output_file = os.path.join(output_folder, f"{file_name.replace('.csv', '.nc')}")
            ds.to_netcdf(output_file)
            print(f"File {file_name} dikonversi dan disimpan ke {output_file}")
        except Exception as e:
            print(f"Error processing {file_name}: {str(e)}")

### 2.2 NetCDF → CSV

In [ ]:
# NC TO CSV
import xarray as xr
import pandas as pd
import os
# Path folder input dan output
input_folder= "/content/drive/MyDrive/regrid(yanto)"
output_folder = "/content/drive/MyDrive/regrid(yanto)/csv"
os.makedirs(output_folder, exist_ok=True)

def process_nc_file(file_path):
    ds = xr.open_dataset(file_path)
    time_var = 'time'

    if time_var not in ds:
        raise ValueError(f"Variabel waktu tidak ditemukan di {file_path}")

    ds[time_var] = pd.to_datetime(ds[time_var].values)
    df_list = []

    for time in ds[time_var].values:
        datetime_object = pd.Timestamp(time)
        year_month = datetime_object.strftime("%Y-%m")
        ds_filtered = ds.sel(time=time)
        df = ds_filtered.to_dataframe().reset_index()
        #df = df.rename(columns={'vimdf': 'VIMFC'})
        #df['VIMFC'] = df['VIMFC'] *-1
        df = df.drop(columns=['pressure_level', 'number'], errors='ignore')
        df_list.append(df)

    df_final = pd.concat(df_list, ignore_index=True)
    return df_final

# Loop semua file dalam folder
for file_name in os.listdir(input_folder):
    if file_name.endswith(".nc"):
        file_path = os.path.join(input_folder, file_name)
        df_final = process_nc_file(file_path)
        output_file = os.path.join(output_folder, f"{file_name.replace('.nc', '.csv')}")
        df_final.to_csv(output_file, index=False)
        print(f"File {file_name} dikonversi dan disimpan ke {output_file}")

### 2.3 Memisahkan Data per Bulan

In [ ]:
import os
import pandas as pd
from datetime import datetime

def process_persiann_by_month(input_file, output_folder):
    """
    Memproses file PERSIANN dan memisahkan data per bulan

    Args:
        input_file: Path ke file CSV PERSIANN
        output_folder: Folder untuk menyimpan hasil per bulan
    """
    # Buat folder output jika belum ada
    os.makedirs(output_folder, exist_ok=True)

    try:
        # Baca file PERSIANN
        df = pd.read_csv(input_file)

        # Cek kolom yang diperlukan
        if 'time' not in df.columns:
            raise ValueError("File harus mengandung kolom 'date'")

        # Konversi kolom date ke datetime
        df['time'] = pd.to_datetime(df['time'])

        # Ekstrak bulan dari tanggal
        df['month'] = df['time'].dt.month

        # Pisahkan data per bulan
        for month in range(1, 13):
            month_data = df[df['month'] == month]

            if not month_data.empty:
                # Nama file output (contoh: PERSIANN_2013_01.csv)
                output_file = os.path.join(output_folder, f'2014{month:02d}.csv')

                # Simpan data bulanan
                month_data.to_csv(output_file, index=False)
                print(f"Data bulan {month:02d} disimpan ke {output_file}")
            else:
                print(f"Tidak ada data untuk bulan {month:02d}")

        print("\nProses pemisahan data per bulan selesai!")

    except Exception as e:
        print(f"Error: {e}")

# Contoh penggunaan
if __name__ == "__main__":
    # Sesuaikan path berikut dengan lokasi file Anda
    input_file = "/content/drive/MyDrive/IDW_(YANTO)/data_2014.csv"
    output_folder = "/content/drive/MyDrive/IDW_(YANTO)/2014"

    process_persiann_by_month(input_file, output_folder)

## 3. Regridding
Data PERSIANN dan MSWEP memiliki resolusi spasial yang berbeda. Sebelum bisa dibandingkan atau digabung, keduanya diregrid ke grid target yang sama di area Pulau Jawa (resolusi 0.125°) menggunakan interpolasi *nearest neighbor*, dengan cropping area terlebih dahulu supaya proses lebih efisien.

In [ ]:
import xarray as xr
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)

INPUT_FOLDER = Path("/content/drive/MyDrive/PERSIANN_Validation_cara2_NC")
OUTPUT_FOLDER = Path("/content/drive/MyDrive/regrid(yanto)v2")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

REGION = {
    'lon_min': 105.9375,
    'lon_max': 114.5625,
    'lat_min': -8.6875,
    'lat_max': -6.0625
}
RESOLUTION = 0.125
CROP_MARGIN = 1.0

def generate_grid(region, resolution):
    lon_start = region['lon_min']
    lon_end = region['lon_max']
    lat_start = region['lat_min']
    lat_end = region['lat_max']
    new_lon = np.round(np.arange(lon_start, lon_end + resolution/2, resolution), 6)
    new_lat = np.round(np.arange(lat_start, lat_end + resolution/2, resolution), 6)
    if new_lat[0] > new_lat[-1]:
        new_lat = new_lat[::-1]
    return new_lon, new_lat

def regrid_xarray_nearest_fill(input_path, output_path):
    try:
        with xr.open_dataset(input_path) as ds:
            if ds.lat[0] > ds.lat[-1]:
                ds = ds.sortby('lat')
            ds_cropped = ds.sel(
                lon=slice(REGION['lon_min'] - CROP_MARGIN, REGION['lon_max'] + CROP_MARGIN),
                lat=slice(REGION['lat_min'] - CROP_MARGIN, REGION['lat_max'] + CROP_MARGIN)
            )
            new_lon, new_lat = generate_grid(REGION, RESOLUTION)
            # Interpolasi nearest
            ds_regrid = ds_cropped.interp(
                lon=new_lon,
                lat=new_lat,
                method="nearest",
                assume_sorted=True
            )
            # Jika masih ada NaN, isi dengan nearest fill
            if np.isnan(ds_regrid.to_array()).sum().item() > 0:
                for var in ds_regrid.data_vars:
                    ds_regrid[var] = ds_regrid[var].interpolate_na(dim="lon", method="nearest", fill_value="extrapolate")
                    ds_regrid[var] = ds_regrid[var].interpolate_na(dim="lat", method="nearest", fill_value="extrapolate")
            ds_regrid.to_netcdf(output_path)
            return True
    except Exception as e:
        print(f"\nError processing {input_path.name}: {e}")
        return False

def main():
    nc_files = sorted(INPUT_FOLDER.glob("*.nc"))
    if not nc_files:
        print(f"No .nc files found in {INPUT_FOLDER}")
        return
    print(f"Found {len(nc_files)} files to process")
    success_count = 0
    for nc_file in tqdm(nc_files, desc="Processing files"):
        output_file = OUTPUT_FOLDER / nc_file.name
        if regrid_xarray_nearest_fill(nc_file, output_file):
            success_count += 1
    print(f"\nCompleted! Successfully processed {success_count}/{len(nc_files)} files")
    print(f"Output saved to: {OUTPUT_FOLDER}")

if __name__ == "__main__":
    main()

## 4. Bias Correction Data PERSIANN (untuk IDW1 & MLP)
Data stasiun PERSIANN dikoreksi bias terhadap data referensi MSWEP **sebelum** diinterpolasi. Hasil koreksi ini (`persiann_corrected`) menjadi input untuk **IDW1** dan **MLP**.

In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from datetime import datetime

def load_persiann_data(filepath):
    data = pd.read_csv(filepath)
    # Pastikan kolom precip ada, rename ke persiann_value
    if 'precip' in data.columns:
        data = data.rename(columns={'precip': 'persiann_value'})
    return data[['lat', 'lon', 'persiann_value']]

def load_mswep_data(filepath):
    ds = xr.open_dataset(filepath)
    # Misal variabel presipitasi bernama 'precipitation'
    # Extract lat, lon, precip jadi DataFrame
    df = ds.to_dataframe().reset_index()
    df = df.rename(columns={'precipitation': 'mswep_value'})
    return df[['lat', 'lon', 'mswep_value']]

def calculate_quantile_mapping(persiann_vals, mswep_vals):
    # Filter nan
    valid = (~np.isnan(persiann_vals)) & (~np.isnan(mswep_vals))
    persiann_vals = persiann_vals[valid]
    mswep_vals = mswep_vals[valid]

    persiann_sorted = np.sort(persiann_vals)
    mswep_sorted = np.sort(mswep_vals)

    def quantile_map(x):
        rank = np.searchsorted(persiann_sorted, x, side='right') / len(persiann_sorted)
        return np.interp(rank, np.linspace(0, 1, len(mswep_sorted)), mswep_sorted)

    return quantile_map

def cross_validate_with_coords(persiann_data, mswep_data, method='quantile'):
    # Gabungkan berdasarkan koordinat lat, lon
    merged = pd.merge(
        persiann_data,
        mswep_data,
        on=['lat', 'lon'],
        how='inner'
    ).dropna(subset=['persiann_value', 'mswep_value'])

    persiann_vals = merged['persiann_value'].values
    mswep_vals = merged['mswep_value'].values

    if method == 'quantile':
        quantile_map = calculate_quantile_mapping(persiann_vals, mswep_vals)
        corrected = quantile_map(persiann_vals)
    else:
        # Misal metode lain, misal IDW, bisa ditambahkan
        corrected = persiann_vals  # sementara tanpa koreksi

    results = merged.copy()
    results['persiann_corrected'] = corrected
    results['correction_method'] = method

    stats = {
        'r2': r2_score(results['mswep_value'], results['persiann_corrected']),
        'rmse': np.sqrt(mean_squared_error(results['mswep_value'], results['persiann_corrected'])),
        'mae': mean_absolute_error(results['mswep_value'], results['persiann_corrected']),
        'n_samples': len(results)
    }

    return results, stats

def plot_quantile_comparison(persiann, mswep, corrected, output_path):
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=persiann, y=mswep, label='Asli', alpha=0.5)
    sns.scatterplot(x=persiann, y=corrected, label='Terkoreksi', alpha=0.5)
    plt.plot([0, max(persiann)], [0, max(persiann)], '--k', label='1:1')
    plt.xlabel('PERSIANN Precipitation (mm)')
    plt.ylabel('MSWEP Precipitation (mm)')
    plt.title('Perbandingan Quantile Sebelum dan Sesudah Koreksi')
    plt.legend()
    plt.savefig(output_path)
    plt.close()

def extract_datetime_from_filename(filename):
    """Ekstrak datetime dari nama file seperti 201301 jadi 2013-1-1."""
    base = os.path.splitext(filename)[0]
    if len(base) >= 6 and base[:6].isdigit():
        year = int(base[:4])
        month = int(base[4:6])
        dt = datetime(year, month, 1)
        return dt.strftime('%Y-%-m-%-d') if hasattr(dt, 'strftime') else f"{year}-{month}-1"
    return ""

def main(persiann_folder, mswep_folder, output_folder, method='quantile'):
    os.makedirs(output_folder, exist_ok=True)

    persiann_files = [f for f in os.listdir(persiann_folder) if f.endswith('.csv')]
    mswep_files = {os.path.splitext(f)[0]: f for f in os.listdir(mswep_folder) if f.endswith('.nc')}

    for persiann_file in persiann_files:
        base_name = os.path.splitext(persiann_file)[0]
        if base_name in mswep_files:
            mswep_file = mswep_files[base_name]

            print(f"\nProcessing {persiann_file} with {mswep_file} using {method} correction...")

            persiann_data = load_persiann_data(os.path.join(persiann_folder, persiann_file))
            mswep_data = load_mswep_data(os.path.join(mswep_folder, mswep_file))

            results, stats = cross_validate_with_coords(persiann_data, mswep_data, method=method)

            # Tambah kolom datetime
            dt_str = extract_datetime_from_filename(persiann_file)
            results['datetime'] = dt_str

            # Save output
            results_file = os.path.join(output_folder, f"{base_name}_validation_results.csv")
            stats_file = os.path.join(output_folder, f"{base_name}_validation_stats.csv")
            plot_file = os.path.join(output_folder, f"{base_name}_quantile_comparison.png")

            results.to_csv(results_file, index=False)
            pd.Series(stats).to_csv(stats_file)

            if method == 'quantile' and not results.empty:
                plot_quantile_comparison(
                    persiann=results['persiann_value'].values,
                    mswep=results['mswep_value'].values,
                    corrected=results['persiann_corrected'].values,
                    output_path=plot_file
                )

            print("=== Validation Summary ===")
            print(stats)

        else:
            print(f"WARNING: No matching MSWEP file found for {persiann_file}")

if __name__ == "__main__":
    main(
        persiann_folder="/content/drive/MyDrive/PERSIANN/2014",
        mswep_folder="/content/drive/MyDrive/0.25_MSWEP/",
        output_folder="/content/drive/MyDrive/PERSIANN_Validation/",
        method='quantile'
    )

## 5. IDW1 — Interpolasi Data yang Sudah Dikoreksi Bias
IDW1 menginterpolasi data PERSIANN yang **sudah dikoreksi bias** (`persiann_corrected`). Parameter **power (p)** dan **radius pencarian** dicari nilai optimalnya per bulan menggunakan 5-fold cross-validation. Hasil akhir skripsi: **power optimal = 3, radius optimal = 30 km**, dengan RMSE ≈ **18,93 mm**.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from sklearn.model_selection import KFold
from PIL import Image

def idw(x, y, values, xi, yi, radius_km=50, p=2):
    radius_deg = radius_km / 111.0
    tree = cKDTree(np.c_[x, y])
    xi_flat = xi.ravel()
    yi_flat = yi.ravel()
    zi = np.full(xi_flat.shape, np.nan)
    for i, (xq, yq) in enumerate(zip(xi_flat, yi_flat)):
        dists, idx = tree.query([xq, yq], k=len(x), distance_upper_bound=radius_deg)
        mask = np.isfinite(dists) & (dists < radius_deg)
        dists = dists[mask]
        idx = idx[mask]
        if len(idx) == 0:
            continue
        dists[dists == 0] = 1e-10
        weights = 1 / (dists ** p)
        weights /= weights.sum()
        zi[i] = np.sum(weights * values[idx])
    return zi.reshape(xi.shape)

def idw_leave_one_out(x, y, values, radius_km=50, p=2):
    n = len(x)
    preds = np.full(n, np.nan)
    for i in range(n):
        mask = np.ones(n, dtype=bool)
        mask[i] = False
        if np.sum(mask) < 1:
            continue
        preds[i] = idw(x[mask], y[mask], values[mask],
                       np.array([x[i]]), np.array([y[i]]),
                       radius_km=radius_km, p=p)[0]
    return preds

def read_pgw(pgw_path):
    with open(pgw_path, 'r') as f:
        lines = [float(line.strip()) for line in f.readlines()]
    return lines  # [A, D, B, E, C, F]

def lonlat_to_pixel(lon, lat, A, E, C, F):
    x_pix = (lon - C) / A
    y_pix = (lat - F) / E
    return x_pix, y_pix

def compute_pod_far_csi(obs, pred, threshold):
    hits = np.sum((obs >= threshold) & (pred >= threshold))
    misses = np.sum((obs >= threshold) & (pred < threshold))
    false_alarms = np.sum((obs < threshold) & (pred >= threshold))
    correct_negatives = np.sum((obs < threshold) & (pred < threshold))
    POD = hits / (hits + misses) if (hits + misses) > 0 else np.nan
    FAR = false_alarms / (hits + false_alarms) if (hits + false_alarms) > 0 else np.nan
    CSI = hits / (hits + misses + false_alarms) if (hits + misses + false_alarms) > 0 else np.nan
    return POD, FAR, CSI, hits, misses, false_alarms, correct_negatives

def main():
    input_folder = "/content/drive/MyDrive/PERSIANN_Validation"
    output_dir = "/content/drive/MyDrive/IDW(PERSIANN)v1/bulanan_revisi"
    os.makedirs(output_dir, exist_ok=True)

    # --- Topografi PNG+PGW ---
    topo_png_path = "/content/drive/MyDrive/JAWAAAA/JAWA.png"
    topo_pgw_path = "/content/drive/MyDrive/JAWAAAA/JAWA.pgw"
    # Load topografi
    A, D, B, E, C, F = read_pgw(topo_pgw_path)
    img = Image.open(topo_png_path)
    img_array = np.asarray(img)
    height, width = img_array.shape[:2]

    # --- Buat mask daratan dari PNG ---
    if img_array.ndim == 3:
        gray = img_array.mean(axis=2)
    else:
        gray = img_array
    mask_land = gray < 250  # threshold, sesuaikan jika perlu

    csv_files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]
    if not csv_files:
        print("Folder input tidak ada file CSV.")
        return

    radius_candidates = [50, 75, 100, 125, 150]
    p_candidates = [0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5]

    min_lat, max_lat = -8.6875, -6.0625
    min_lon, max_lon = 105.1875, 114.5625
    grid_res = 0.125

    gridx = np.arange(min_lon, max_lon + grid_res, grid_res)
    gridy = np.arange(min_lat, max_lat + grid_res, grid_res)
    gridx_proj, gridy_proj = np.meshgrid(gridx, gridy)

    # --- Kumpulkan seluruh data darat dari semua CSV untuk threshold global ---
    all_values_land = []
    for csv_file in csv_files:
        data = pd.read_csv(os.path.join(input_folder, csv_file))
        if not all(col in data.columns for col in ['lon', 'lat', 'persiann_corrected']):
            continue
        x = data['lon'].values
        y = data['lat'].values
        values = data['persiann_corrected'].values
        x_pts_pixel, y_pts_pixel = lonlat_to_pixel(x, y, A, E, C, F)
        on_land = []
        for xp, yp in zip(x_pts_pixel, y_pts_pixel):
            xi, yi = int(round(xp)), int(round(yp))
            if (0 <= xi < mask_land.shape[1]) and (0 <= yi < mask_land.shape[0]) and mask_land[yi, xi]:
                on_land.append(True)
            else:
                on_land.append(False)
        on_land = np.array(on_land)
        all_values_land.append(values[on_land])
    all_values_land = np.concatenate(all_values_land)
    perc95_rf = np.percentile(all_values_land, 95)
    print(f"Threshold 95% dari seluruh data darat: {perc95_rf:.2f} mm")

    eval_list = []

    for csv_file in csv_files:
        print(f"\nMemproses file {csv_file}...")
        data = pd.read_csv(os.path.join(input_folder, csv_file))
        if 'datetime' not in data.columns:
            print(f"File {csv_file} dilewati (tidak ada kolom 'datetime').")
            continue

        data['datetime'] = pd.to_datetime(data['datetime'])
        data['month'] = data['datetime'].dt.month
        data['year'] = data['datetime'].dt.year

        for (year, month), monthly_data in data.groupby(['year', 'month']):
            print(f"\nMemproses {year}-{month}...")
            x = monthly_data['lon'].values
            y = monthly_data['lat'].values
            values = monthly_data['persiann_corrected'].values

            # --- Filter titik darat dulu untuk threshold dan mask ---
            x_pts_pixel, y_pts_pixel = lonlat_to_pixel(x, y, A, E, C, F)
            on_land = []
            for xp, yp in zip(x_pts_pixel, y_pts_pixel):
                xi, yi = int(round(xp)), int(round(yp))
                if (0 <= xi < mask_land.shape[1]) and (0 <= yi < mask_land.shape[0]) and mask_land[yi, xi]:
                    on_land.append(True)
                else:
                    on_land.append(False)
            on_land = np.array(on_land)
            x_land = x[on_land]
            y_land = y[on_land]
            values_land = values[on_land]

            if len(x_land) < 1:
                print(f"{year}-{month}: Tidak ada data stasiun di daratan, dilewati.")
                continue

            # --- Pakai threshold global dari seluruh data darat ---
            threshold_eval = perc95_rf
            print(f"Threshold 95% (global): {threshold_eval:.2f} mm")

            # --- Interpolasi IDW (pakai semua data, atau bisa juga hanya darat) ---
            try:
                optimal_radius, optimal_p = find_optimal_radius_and_p(
                    x, y, values,
                    radius_values=radius_candidates,
                    p_values=p_candidates
                )
            except:
                optimal_radius, optimal_p = 50, 2

            z = idw(x, y, values, gridx_proj, gridy_proj, radius_km=optimal_radius, p=optimal_p)

            # --- Simpan hasil grid interpolasi ke CSV ---
            result_df = pd.DataFrame({
                'lat': gridy_proj.ravel(),
                'lon': gridx_proj.ravel(),
                'rain': z.ravel()
            })
            out_csv_path = os.path.join(
                output_dir, f'{os.path.splitext(csv_file)[0]}_{year}_{month}_idw.csv'
            )
            result_df.to_csv(out_csv_path, index=False)
            print(f"Hasil IDW {year}-{month} disimpan di {out_csv_path}")

            # --- Masking curah hujan di daratan saja ---
            xpix, ypix = lonlat_to_pixel(gridx_proj, gridy_proj, A, E, C, F)
            mask_grid = np.zeros_like(z, dtype=bool)
            for i in range(z.shape[0]):
                for j in range(z.shape[1]):
                    xi, yi = int(round(xpix[i, j])), int(round(ypix[i, j]))
                    if (0 <= xi < mask_land.shape[1]) and (0 <= yi < mask_land.shape[0]):
                        mask_grid[i, j] = mask_land[yi, xi]
                    else:
                        mask_grid[i, j] = False
            z_masked = np.ma.masked_where(~mask_grid, z)

            # --- Titik threshold: hanya jika grid IDW diplot pada layer curah hujan (mask_grid True dan nilai > 0) ---
            mask_95 = (values > threshold_eval) & on_land
            x_thresh, y_thresh = x[mask_95], y[mask_95]
            x_thresh_pix, y_thresh_pix = lonlat_to_pixel(x_thresh, y_thresh, A, E, C, F)
            valid_titik = []
            for lon0, lat0, xp, yp in zip(x_thresh, y_thresh, x_thresh_pix, y_thresh_pix):
                # Cari grid IDW terdekat
                dist = (gridx_proj - lon0)**2 + (gridy_proj - lat0)**2
                idx = np.unravel_index(np.argmin(dist), dist.shape)
                val_here = z_masked[idx]
                if mask_grid[idx] and (not np.ma.is_masked(val_here)) and (not np.isnan(val_here)) and (val_here > 0):
                    xi, yi = int(round(xp)), int(round(yp))
                    if (0 <= xi < mask_land.shape[1]) and (0 <= yi < mask_land.shape[0]):
                        valid_titik.append((xp, yp))
            valid_titik = np.array(valid_titik)

            # --- Plotting tanpa colorbar ---
            fig, ax = plt.subplots(figsize=(16, 6))
            ax.imshow(img_array, origin='upper')
            rain = ax.pcolormesh(xpix, ypix, z_masked, shading='auto', alpha=0.5, cmap='viridis', vmin=0, vmax=800)
            # plt.colorbar(rain, ax=ax, label='Curah Hujan (mm)')  # HAPUS colorbar
            if len(valid_titik) > 0:
                ax.scatter(valid_titik[:,0], valid_titik[:,1], color='white', s=30, marker='s', zorder=5)
            ax.set_xlim(0, img_array.shape[1])
            ax.set_ylim(img_array.shape[0], 0)
            plt.title(f'IDW1 {year}-{month} (Thres95={threshold_eval:.1f} mm)')
            plt.tight_layout()
            png_path = os.path.join(output_dir, f'{os.path.splitext(csv_file)[0]}_{year}_{month}_thres95_layeronly.png')
            plt.savefig(png_path, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"Plot {year}-{month} disimpan di {png_path}")

            # --- Evaluasi: POD, FAR, CSI dengan leave-one-out ---
            pred_obs = idw_leave_one_out(x_land, y_land, values_land, radius_km=optimal_radius, p=optimal_p)
            obs = values_land
            pred = pred_obs

            pod, far, csi, hits, misses, false_alarms, correct_negatives = compute_pod_far_csi(
                obs, pred, threshold_eval
            )
            eval_dict = {
                'csv_file': csv_file,
                'year': year,
                'month': month,
                'thres95': threshold_eval,
                'POD': pod,
                'FAR': far,
                'CSI': csi,
                'hits': hits,
                'misses': misses,
                'false_alarms': false_alarms,
                'correct_negatives': correct_negatives,
                'n_obs': len(obs)
            }
            eval_list.append(eval_dict)

    # --- Simpan hasil evaluasi ke CSV (semua jadi satu file) ---
    if eval_list:
        eval_df = pd.DataFrame(eval_list)
        eval_csv_path = os.path.join(output_dir, 'hasil_POD_FAR_CSI_bulanan.csv')
        eval_df.to_csv(eval_csv_path, index=False)
        print(f"Rekap evaluasi disimpan di {eval_csv_path}")

if __name__ == "__main__":
    main()

## 6. IDW2 — Interpolasi Data Mentah, lalu Koreksi Bias pada Hasil
IDW2 menempuh urutan terbalik: data PERSIANN **mentah** diinterpolasi terlebih dahulu (dengan power & radius optimal yang sama, hasil RMSE ≈ **18,71 mm**), baru kemudian **hasil interpolasinya** dikoreksi bias terhadap MSWEP menggunakan *quantile mapping*.

### 6.1 Interpolasi IDW pada data mentah

In [ ]:
#rata2 bulanan parameter optimal
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from PIL import Image
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import csv
import collections

def read_pgw(pgw_path):
    with open(pgw_path, 'r') as f:
        lines = [float(line.strip()) for line in f.readlines()]
    return lines  # [A, D, B, E, C, F]

def lonlat_to_pixel(lon, lat, A, E, C, F):
    x_pix = (lon - C) / A
    y_pix = (lat - F) / E
    return x_pix, y_pix

def pixel_to_lonlat(x_pix, y_pix, A, E, C, F):
    lon = x_pix * A + C
    lat = y_pix * E + F
    return lon, lat

def idw(x, y, values, xi, yi, radius_km=50, p=2):
    radius_deg = radius_km / 111.0
    tree = cKDTree(np.c_[x, y])
    xi_flat = xi.ravel()
    yi_flat = yi.ravel()
    zi = np.full(xi_flat.shape, np.nan)
    for i, (xq, yq) in enumerate(zip(xi_flat, yi_flat)):
        dists, idx = tree.query([xq, yq], k=len(x), distance_upper_bound=radius_deg)
        mask = np.isfinite(dists) & (dists < radius_deg)
        dists = dists[mask]
        idx = idx[mask]
        if len(idx) == 0:
            continue
        dists[dists == 0] = 1e-10
        weights = 1 / (dists ** p)
        weights /= weights.sum()
        zi[i] = np.sum(weights * values[idx])
    return zi.reshape(xi.shape)

def idw_kfold_cv(x, y, values, radius_km, p, n_splits=5, random_state=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    errors = []
    coords = np.array([x, y]).T
    for train_idx, test_idx in kf.split(coords):
        x_train, y_train, v_train = x[train_idx], y[train_idx], values[train_idx]
        x_test, y_test, v_test = x[test_idx], y[test_idx], values[test_idx]
        pred = []
        for xi, yi in zip(x_test, y_test):
            radius_deg = radius_km / 111.0
            tree = cKDTree(np.c_[x_train, y_train])
            dists, idx = tree.query([xi, yi], k=len(x_train), distance_upper_bound=radius_deg)
            mask = np.isfinite(dists) & (dists < radius_deg)
            dists = dists[mask]
            idx = idx[mask]
            if len(idx) == 0:
                pred.append(np.nan)
                continue
            dists[dists == 0] = 1e-10
            weights = 1 / (dists ** p)
            weights /= weights.sum()
            pred.append(np.sum(weights * v_train[idx]))
        mask_valid = ~np.isnan(pred)
        if np.sum(mask_valid) == 0:
            continue
        rmse = np.sqrt(mean_squared_error(v_test[mask_valid], np.array(pred)[mask_valid]))
        errors.append(rmse)
    if len(errors) == 0:
        return np.nan
    return np.mean(errors)

def find_best_idw_params(x, y, values, grid_powers, grid_radii, n_splits=5, csv_path=None, return_dict=False):
    best_rmse = np.inf
    best_p = None
    best_r = None
    rmse_dict = {}
    if csv_path is not None:
        csvfile = open(csv_path, 'w', newline='')
        writer = csv.writer(csvfile)
        writer.writerow(["power", "radius_km", "rmse"])
    else:
        writer = None

    for p in grid_powers:
        for r in grid_radii:
            rmse = idw_kfold_cv(x, y, values, r, p, n_splits=n_splits)
            print(f"    power={p}, radius={r}  --> RMSE={rmse:.3f}")
            if writer is not None:
                writer.writerow([p, r, rmse])
            rmse_dict[(p, r)] = rmse
            if rmse < best_rmse:
                best_rmse = rmse
                best_p = p
                best_r = r

    if writer is not None:
        csvfile.close()

    print(f"    Best params: power={best_p}, radius={best_r} (RMSE={best_rmse:.3f})")
    if return_dict:
        return best_p, best_r, rmse_dict
    else:
        return best_p, best_r

def main():
    input_folder = "/content/drive/MyDrive/PERSIANN"
    output_dir = "/content/drive/MyDrive/IDW(PERSIANN)/climatology_subplot2"
    os.makedirs(output_dir, exist_ok=True)

    topo_png_path = "/content/drive/MyDrive/JAWAAAA/JAWA.png"
    topo_pgw_path = "/content/drive/MyDrive/JAWAAAA/JAWA.pgw"
    A, D, B, E, C, F = read_pgw(topo_pgw_path)
    img = Image.open(topo_png_path)
    img_array = np.asarray(img)
    if img_array.ndim == 3:
        gray = img_array.mean(axis=2)
    else:
        gray = img_array
    mask_land = gray < 250

    csv_files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]
    if not csv_files:
        print("Folder input tidak ada file CSV.")
        return

    grid_res = 0.125
    min_lat, max_lat = -8.6875, -6.0625
    min_lon, max_lon = 105.1875, 114.5625
    gridx = np.arange(min_lon, max_lon + grid_res, grid_res)
    gridy = np.arange(min_lat, max_lat + grid_res, grid_res)
    gridx_proj, gridy_proj = np.meshgrid(gridx, gridy)
    xpix, ypix = lonlat_to_pixel(gridx_proj, gridy_proj, A, E, C, F)

    monthly_grids = {m: [] for m in range(1, 13)}
    mask_grid_template = None

    grid_powers = np.arange(1.0, 3.01, 0.5)  # 1.0, 1.5, ..., 3.0
    grid_radii = [30, 50, 70]                # in km

    all_rmses = collections.defaultdict(list) # {(p, r): [rmse, ...]}

    for csv_file in csv_files:
        print(f"\nMemproses file {csv_file}...")
        data = pd.read_csv(os.path.join(input_folder, csv_file))
        if 'datetime' not in data.columns:
            print(f"File {csv_file} dilewati (tidak ada kolom 'datetime').")
            continue
        data['datetime'] = pd.to_datetime(data['datetime'])
        data['month'] = data['datetime'].dt.month
        data['year'] = data['datetime'].dt.year

        for (year, month), monthly_data in data.groupby(['year', 'month']):
            x = monthly_data['lon'].values
            y = monthly_data['lat'].values
            values = monthly_data['precip'].values

            x_pts_pixel, y_pts_pixel = lonlat_to_pixel(x, y, A, E, C, F)
            on_land = []
            for xp, yp in zip(x_pts_pixel, y_pts_pixel):
                xi, yi = int(round(xp)), int(round(yp))
                if (0 <= xi < mask_land.shape[1]) and (0 <= yi < mask_land.shape[0]) and mask_land[yi, xi]:
                    on_land.append(True)
                else:
                    on_land.append(False)
            on_land = np.array(on_land)
            x_land = x[on_land]
            y_land = y[on_land]
            values_land = values[on_land]

            if len(x_land) < 5:
                continue

            print(f"  Cari power dan radius optimal (year={year}, month={month}) ...")
            csv_rmse_path = os.path.join(
                output_dir, f"rmse_power_radius_{year}_{month}.csv"
            )
            best_p, best_r, rmse_dict = find_best_idw_params(
                x_land, y_land, values_land, grid_powers, grid_radii, n_splits=5, csv_path=csv_rmse_path, return_dict=True
            )
            # Simpan seluruh rmse ke dict global
            for (p, r), v in rmse_dict.items():
                if not np.isnan(v):
                    all_rmses[(p, r)].append(v)

            z = idw(x_land, y_land, values_land, gridx_proj, gridy_proj, radius_km=best_r, p=best_p)

            mask_grid = np.zeros_like(z, dtype=bool)
            for i in range(z.shape[0]):
                for j in range(z.shape[1]):
                    xi, yi = int(round(xpix[i, j])), int(round(ypix[i, j]))
                    if (0 <= xi < mask_land.shape[1]) and (0 <= yi < mask_land.shape[0]):
                        mask_grid[i, j] = mask_land[yi, xi]
                    else:
                        mask_grid[i, j] = False
            z_masked = np.ma.masked_where(~mask_grid, z)

            monthly_grids[month].append(z_masked.copy())
            if mask_grid_template is None:
                mask_grid_template = mask_grid.copy()

    # Setelah seluruh loop: Hitung rata-rata RMSE tiap kombinasi
    avg_csv_path = os.path.join(output_dir, "avg_rmse_power_radius.csv")
    with open(avg_csv_path, "w", newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["power", "radius_km", "rmse_avg", "rmse_n"])
        for (p, r), rmse_list in sorted(all_rmses.items()):
            rmse_avg = sum(rmse_list) / len(rmse_list)
            writer.writerow([p, r, rmse_avg, len(rmse_list)])
    print(f"Rata-rata RMSE tiap power-radius disimpan di: {avg_csv_path}")

    # --- Hitung rata-rata bulanan (climatology) ---
    monthly_mean = {}
    for m in range(1, 13):
        if monthly_grids[m]:
            stacked = np.ma.stack(monthly_grids[m])
            monthly_mean[m] = np.ma.mean(stacked, axis=0)
        else:
            monthly_mean[m] = np.ma.masked_all(mask_grid_template.shape)

    # --- Simpan ke CSV ---
    print("Menyimpan ke CSV...")
    with open(os.path.join(output_dir, 'climatology_12bulan.csv'), 'w') as f:
        f.write('lat,lon,month,rain\n')
        for m in range(1, 13):
            zmean = monthly_mean[m]
            for i in range(zmean.shape[0]):
                for j in range(zmean.shape[1]):
                    if not np.ma.is_masked(zmean[i, j]):
                        f.write(f"{gridy_proj[i, j]},{gridx_proj[i, j]},{m},{zmean[i, j]}\n")

    # --- Plot subplot 3x4 + colorbar manual ---
    import matplotlib.gridspec as gridspec
    n_rows, n_cols = 3, 4
    fig_width = 6.5 * n_cols
    fig_height = 3 * n_rows
    fig = plt.figure(figsize=(fig_width, fig_height))
    gs = gridspec.GridSpec(n_rows, n_cols, wspace=0.18, hspace=0.01)

    axes = []
    months_str = ["Januari", "Februari", "Maret", "April", "Mei", "Juni",
                  "Juli", "Agustus", "September", "Oktober", "November", "Desember"]
    vmin, vmax = 0, 800
    cmap = 'viridis'
    rain_for_cbar = None

    # XTICKS/YTICKS
    n_xticks, n_yticks = 5, 5
    xtick_pix = np.linspace(0, img_array.shape[1], n_xticks)
    ytick_pix = np.linspace(img_array.shape[0], 0, n_yticks)
    xtick_lon = []
    for tick in xtick_pix:
        lon, _ = pixel_to_lonlat(tick, 0, A, E, C, F)
        xtick_lon.append(lon)
    ytick_lat = []
    for tick in ytick_pix:
        _, lat = pixel_to_lonlat(0, tick, A, E, C, F)
        ytick_lat.append(lat)

    for idx in range(12):
        row, col = divmod(idx, 4)
        ax = fig.add_subplot(gs[row, col])
        axes.append(ax)
        m = idx + 1
        zmean = monthly_mean[m]
        ax.imshow(img_array, origin='upper', zorder=1)
        rain = ax.pcolormesh(xpix, ypix, zmean, shading='auto', alpha=0.5,
                             cmap=cmap, vmin=vmin, vmax=vmax, zorder=2)
        ax.set_xlim(0, img_array.shape[1])
        ax.set_ylim(img_array.shape[0], 0)
        ax.set_title(months_str[idx], fontsize=13)
        ax.set_xticks(xtick_pix)
        ax.set_yticks(ytick_pix)
        ax.set_xticklabels([f"{lon:.2f}" for lon in xtick_lon], fontsize=10)
        ax.set_yticklabels([f"{lat:.2f}" for lat in ytick_lat], fontsize=10)
        ax.set_xlabel('Longitude', fontsize=11)
        ax.set_ylabel('Latitude', fontsize=11)
        ax.tick_params(axis='both', which='both', length=2.5, labelsize=9)
        ax.grid(which='both', linestyle=':', color='k', alpha=0.4, zorder=3)
        for direction in ['left', 'right', 'top', 'bottom']:
            ax.spines[direction].set_linewidth(2.5)
            ax.spines[direction].set_color('black')
        if idx == 0:
            rain_for_cbar = rain

    plt.subplots_adjust(left=0.055, right=0.91, bottom=0.08, top=0.93, hspace=0.01, wspace=0.18)

    # Colorbar manual: setinggi subplot paling atas sampai paling bawah
    top_ax = axes[0]
    bottom_ax = axes[(n_rows-1)*n_cols]
    top_pos = top_ax.get_position()
    bottom_pos = bottom_ax.get_position()
    cbar_ax = fig.add_axes([
        0.93,                        # posisi x (kanan plot)
        bottom_pos.y0,               # y bawah
        0.018,                       # lebar
        top_pos.y1 - bottom_pos.y0   # tinggi
    ])
    cb = plt.colorbar(rain_for_cbar, cax=cbar_ax, label='Curah Hujan (mm)')
    cbar_ax.tick_params(labelsize=12)

    plt.suptitle('Rata-rata Bulanan IDW2 Optimal 2001-2014', fontsize=22, y=top_pos.y1 + 0.08)
    plt.savefig(os.path.join(output_dir, 'climatology_12bulan_IDW_optimal_subplot.png'), dpi=300, bbox_inches='tight', pad_inches=0.1)
    plt.show()

if __name__ == "__main__":
    main()

### 6.2 Koreksi bias pada hasil interpolasi IDW (quantile mapping terhadap MSWEP)

In [ ]:
#optimal
import os
import re
import pandas as pd
import xarray as xr
import numpy as np
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt

def load_mswep_data(file_path, lats=None, lons=None):
    ds = xr.open_dataset(file_path)
    precip_var = None
    for var in ds.data_vars:
        if 'precip' in var.lower() or 'tp' in var.lower() or 'rain' in var.lower():
            precip_var = var
            break
    if precip_var is None:
        raise ValueError(f"Precipitation variable not found in {file_path}")
    lat_dim = 'lat' if 'lat' in ds.dims else 'latitude'
    lon_dim = 'lon' if 'lon' in ds.dims else 'longitude'
    if lats is not None and lons is not None:
        ds = ds.sel(
            **{lat_dim: xr.DataArray(lats, dims="points"),
               lon_dim: xr.DataArray(lons, dims="points")},
            method="nearest"
        )
    df = ds[precip_var].to_dataframe().reset_index()
    df = df.rename(columns={precip_var: 'value', lat_dim: 'lat', lon_dim: 'lon'})
    return df

def idw_interpolation(idw_data, mswep_data):
    mean_idw = idw_data['value'].mean()
    mean_mswep = mswep_data['value'].mean()
    scaling_factor = mean_mswep / mean_idw if mean_idw != 0 else 1.0
    corrected_df = idw_data.copy()
    corrected_df['value'] = idw_data['value'] * scaling_factor
    return corrected_df

def quantile_mapping_correction(idw_data, mswep_data):
    min_length = min(len(idw_data), len(mswep_data))
    idw_values = idw_data['value'].values[:min_length]
    mswep_values = mswep_data['value'].values[:min_length]
    idw_sorted = np.sort(idw_values)
    mswep_sorted = np.sort(mswep_values)
    quantiles = np.linspace(0, 1, min_length, endpoint=True)
    mswep_interp = interp1d(quantiles, mswep_sorted, bounds_error=False, fill_value='extrapolate')
    idw_ranks = np.argsort(np.argsort(idw_values)) / (min_length - 1)
    corrected_values = mswep_interp(idw_ranks)
    corrected_df = idw_data.iloc[:min_length].copy()
    corrected_df['value'] = corrected_values
    return corrected_df

def cross_validate_idw(idw_data, mswep_data, method='quantile', datetime_str=None):
    if method == 'idw':
        corrected_data = idw_interpolation(idw_data, mswep_data)
    elif method == 'quantile':
        corrected_data = quantile_mapping_correction(idw_data, mswep_data)
    else:
        raise ValueError(f"Unknown method: {method}")

    min_length = min(len(idw_data), len(mswep_data), len(corrected_data))

    results = pd.DataFrame()
    if datetime_str is not None:
        results['datetime'] = [datetime_str] * min_length
    if 'lat' in idw_data.columns:
        results['lat'] = idw_data['lat'].values[:min_length]
    if 'lon' in idw_data.columns:
        results['lon'] = idw_data['lon'].values[:min_length]
    results['idw_value'] = idw_data['value'].values[:min_length]
    results['mswep_value'] = mswep_data['value'].values[:min_length]
    results['idw_corrected'] = corrected_data['value'].values[:min_length]
    results['correction_method'] = [method] * min_length

    stats = pd.DataFrame({
        'mean_idw': [idw_data['value'].mean()],
        'mean_mswep': [mswep_data['value'].mean()],
        'mean_corrected': [corrected_data['value'].mean()]
    })

    return results, stats, corrected_data

def plot_quantile_comparison(idw, mswep, corrected, output_path):
    plt.figure(figsize=(8, 6))
    plt.scatter(idw, mswep, label='Original IDW vs MSWEP', alpha=0.5)
    plt.scatter(corrected, mswep, label='Corrected IDW vs MSWEP', alpha=0.5)
    plt.xlabel('IDW (mm)')
    plt.ylabel('MSWEP (mm)')
    plt.legend()
    plt.title('Quantile Mapping Comparison')
    plt.grid(True)
    plt.savefig(output_path)
    plt.close()

def main(idw_folder, mswep_folder, output_folder, method='quantile'):
    os.makedirs(output_folder, exist_ok=True)
    idw_files = [f for f in os.listdir(idw_folder) if f.endswith('_idw.nc')]
    mswep_files = {os.path.splitext(f)[0]: f for f in os.listdir(mswep_folder) if f.endswith('.nc')}

    for idw_file in idw_files:
        # Ekstrak tahun dan bulan dari nama file IDW, contoh: 2001_2001_10_idw.nc
        match = re.match(r"(\d{4})_(\d{4})_(\d{1,2})_idw\.nc", idw_file)
        if match:
            year, year2, month = match.groups()
            month_str = f"{int(month):02d}"
            mswep_name = f"{year}{month_str}"  # contoh: 200110
            mswep_file = mswep_files.get(mswep_name)
            print(f"IDW: {idw_file}, expect MSWEP: {mswep_name}.nc")
        else:
            print(f"WARNING: Gagal ekstrak tahun/bulan dari {idw_file}")
            mswep_file = None

        if mswep_file:
            print(f"Processing {idw_file} with {mswep_file} using {method} correction...")

            # Baca data IDW (anggap netCDF, mirip MSWEP)
            idw_data = load_mswep_data(os.path.join(idw_folder, idw_file))
            idw_lats = idw_data['lat'].values
            idw_lons = idw_data['lon'].values

            # Baca data MSWEP dan lakukan spatial matching
            mswep_data = load_mswep_data(
                os.path.join(mswep_folder, mswep_file),
                lats=idw_lats,
                lons=idw_lons
            )

            # Lakukan bias correction
            results, stats, corrected_data = cross_validate_idw(
                idw_data=idw_data,
                mswep_data=mswep_data,
                method=method,
                datetime_str=f"{year}-{month_str}-01"
            )

            base_name = os.path.splitext(idw_file)[0]
            results_file = os.path.join(output_folder, f"{base_name}_validation_results.csv")
            stats_file = os.path.join(output_folder, f"{base_name}_validation_stats.csv")
            corrected_file = os.path.join(output_folder, f"{base_name}_corrected_data.csv")
            plot_file = os.path.join(output_folder, f"{base_name}_quantile_comparison.png")

            results.to_csv(results_file, index=False)
            stats.to_csv(stats_file, index=False)
            corrected_data['datetime'] = f"{year}-{month_str}-01"
            corrected_data.to_csv(corrected_file, index=False)

            if method == 'quantile':
                plot_quantile_comparison(
                    idw=results['idw_value'].values,
                    mswep=results['mswep_value'].values,
                    corrected=results['idw_corrected'].values,
                    output_path=plot_file
                )

            print("\n=== Validation Summary ===")
            print(stats)
        else:
            print(f"WARNING: No matching MSWEP file found for IDW file {idw_file}. Skipping.")

if __name__ == "__main__":
    main(
        idw_folder="/content/drive/MyDrive/IDW(PERSIANN)/topo2",            # folder IDW
        mswep_folder="/content/drive/MyDrive/0.1_MSWEP",                  # folder MSWEP
        output_folder="/content/drive/MyDrive/IDW_Validation2/",           # folder hasil output
        method='quantile'  # atau 'idw'
    )

## 7. MLP — Multilayer Perceptron
MLP dilatih **sekali** menggunakan data PERSIANN yang sudah dikoreksi bias (`persiann_corrected`) — input yang sama dengan IDW1. Arsitektur: 3 hidden layer (150, 75, 30 neuron), *early stopping* untuk mencegah overfitting, validation fraction 15%, learning rate awal 0.0005.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import xarray as xr
from PIL import Image

# One-time convergence warning
import warnings
from sklearn.exceptions import ConvergenceWarning

class OneTimeConvergenceWarning:
    shown = False
    def __call__(self, message, category, filename, lineno, file=None, line=None):
        if not self.shown:
            print(f"{category.__name__}: {message}")
            self.shown = True

warnings.simplefilter("always", category=ConvergenceWarning)
warnings.showwarning = OneTimeConvergenceWarning()

def read_pgw(pgw_path):
    with open(pgw_path, 'r') as f:
        lines = [float(line.strip()) for line in f.readlines()]
    return lines  # [A, D, B, E, C, F]

def lonlat_to_pixel(lon, lat, A, E, C, F):
    x_pix = (lon - C) / A
    y_pix = (lat - F) / E
    return x_pix, y_pix

def mlp_interpolate(x, y, values, xi, yi, scaler, max_iter=1000, random_state=42):
    X_train = np.column_stack([x, y])
    y_train = values
    X_pred = np.column_stack([xi.ravel(), yi.ravel()])
    X_train = scaler.transform(X_train)
    X_pred = scaler.transform(X_pred)
    mlp = MLPRegressor(
        hidden_layer_sizes=(150, 75, 30),
        max_iter=max_iter,
        random_state=random_state,
        early_stopping=True,
        n_iter_no_change=10,
        validation_fraction=0.2,
        learning_rate_init=0.001,
        tol=1e-3,
        verbose=False
    )
    mlp.fit(X_train, y_train)
    z_pred = mlp.predict(X_pred)
    return z_pred.reshape(xi.shape), mlp

def compute_pod_far_csi(obs, pred, threshold):
    hits = np.sum((obs >= threshold) & (pred >= threshold))
    misses = np.sum((obs >= threshold) & (pred < threshold))
    false_alarms = np.sum((obs < threshold) & (pred >= threshold))
    correct_negatives = np.sum((obs < threshold) & (pred < threshold))
    POD = hits / (hits + misses) if (hits + misses) > 0 else np.nan
    FAR = false_alarms / (hits + false_alarms) if (hits + false_alarms) > 0 else np.nan
    CSI = hits / (hits + misses + false_alarms) if (hits + misses + false_alarms) > 0 else np.nan
    return POD, FAR, CSI, hits, misses, false_alarms, correct_negatives

def mlp_kfold_eval(x, y, values, n_splits=5, max_iter=1000, random_state=42):
    n = len(x)
    preds = np.full(n, np.nan)
    kf = KFold(n_splits=min(n_splits, n), shuffle=True, random_state=random_state)
    fold_idx = np.full(n, -1)
    for fold, (train_idx, test_idx) in enumerate(kf.split(x)):
        X_train = np.column_stack([x[train_idx], y[train_idx]])
        y_train = values[train_idx]
        X_test = np.column_stack([x[test_idx], y[test_idx]])
        scaler = StandardScaler().fit(X_train)
        X_train_scaled = scaler.transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        mlp = MLPRegressor(
            hidden_layer_sizes=(150, 75, 30),
            max_iter=max_iter,
            random_state=random_state,
            early_stopping=True,
            n_iter_no_change=10,
            validation_fraction=0.2,
            learning_rate_init=0.001,
            tol=1e-3,
            verbose=False
        )
        try:
            mlp.fit(X_train_scaled, y_train)
            preds[test_idx] = mlp.predict(X_test_scaled)
            fold_idx[test_idx] = fold
        except Exception:
            preds[test_idx] = np.nan
    return preds, fold_idx

def main():
    input_folder = "/content/drive/MyDrive/PERSIANN_Validation"
    output_dir = "/content/drive/MyDrive/MLP(PERSIANN)_Fixed150_75_30_pngmask"
    os.makedirs(output_dir, exist_ok=True)

    # --- Topografi PNG+PGW ---
    topo_png_path = "/content/drive/MyDrive/JAWAAAA/JAWA.png"
    topo_pgw_path = "/content/drive/MyDrive/JAWAAAA/JAWA.pgw"
    A, D, B, E, C, F = read_pgw(topo_pgw_path)
    img = Image.open(topo_png_path)
    img_array = np.asarray(img)
    if img_array.ndim == 3:
        gray = img_array.mean(axis=2)
    else:
        gray = img_array
    mask_land = gray < 250  # threshold sesuai IDW

    # --- Grid domain ---
    min_lat, max_lat = -8.6875, -6.0625
    min_lon, max_lon = 105.1875, 114.5625
    grid_res = 0.125
    gridx = np.arange(min_lon, max_lon + grid_res, grid_res)
    gridy = np.arange(min_lat, max_lat + grid_res, grid_res)
    gridx_proj, gridy_proj = np.meshgrid(gridx, gridy)

    csv_files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]
    if not csv_files:
        print("Folder input tidak ada file CSV.")
        return

    # --- KUMPULKAN SEMUA DATA DI DARAT UNTUK THRESHOLD GLOBAL ---
    all_values_land = []
    for csv_file in csv_files:
        data = pd.read_csv(os.path.join(input_folder, csv_file))
        if 'datetime' not in data.columns:
            continue
        x = data['lon'].values
        y = data['lat'].values
        vals = data['persiann_corrected'].values
        x_pts_pixel, y_pts_pixel = lonlat_to_pixel(x, y, A, E, C, F)
        on_land = []
        for xp, yp in zip(x_pts_pixel, y_pts_pixel):
            xi, yi = int(round(xp)), int(round(yp))
            if (0 <= xi < mask_land.shape[1]) and (0 <= yi < mask_land.shape[0]) and mask_land[yi, xi]:
                on_land.append(True)
            else:
                on_land.append(False)
        on_land = np.array(on_land)
        all_values_land.append(vals[on_land])
    all_values_land = np.concatenate(all_values_land)
    perc95_rf_global = np.percentile(all_values_land, 95)
    print(f"Threshold 95% GLOBAL seluruh data darat: {perc95_rf_global:.2f} mm")

    rmse_records = []
    eval_list = []
    threshold_points_records = []

    for csv_file in csv_files:
        print(f"\nMemproses file {csv_file}...")
        data = pd.read_csv(os.path.join(input_folder, csv_file))
        if 'datetime' not in data.columns:
            print(f"File {csv_file} dilewati (tidak ada kolom 'datetime').")
            continue

        data['datetime'] = pd.to_datetime(data['datetime'])
        data['month'] = data['datetime'].dt.month
        data['year'] = data['datetime'].dt.year

        for (year, month), monthly_data in data.groupby(['year', 'month']):
            print(f"\nMemproses {year}-{month}...")
            x = monthly_data['lon'].values
            y = monthly_data['lat'].values
            values = monthly_data['persiann_corrected'].values
            datetime_value = monthly_data['datetime'].iloc[0]

            # --- Masking titik stasiun di darat dengan PNG/PGW ---
            x_pts_pixel, y_pts_pixel = lonlat_to_pixel(x, y, A, E, C, F)
            on_land = []
            for xp, yp in zip(x_pts_pixel, y_pts_pixel):
                xi, yi = int(round(xp)), int(round(yp))
                if (0 <= xi < mask_land.shape[1]) and (0 <= yi < mask_land.shape[0]) and mask_land[yi, xi]:
                    on_land.append(True)
                else:
                    on_land.append(False)
            on_land = np.array(on_land)
            x_land = x[on_land]
            y_land = y[on_land]
            values_land = values[on_land]

            if len(x_land) < 1:
                print(f"{year}-{month}: Tidak ada data stasiun di daratan, dilewati.")
                continue

            # --- Pakai threshold global ---
            perc95_rf = perc95_rf_global
            print(f"Threshold 95% GLOBAL: {perc95_rf:.2f} mm")

            try:
                scaler = StandardScaler().fit(np.column_stack([x, y]))
                z, mlp = mlp_interpolate(x, y, values, gridx_proj, gridy_proj, scaler=scaler)
                X_self = scaler.transform(np.column_stack([x, y]))
                values_pred = mlp.predict(X_self)
                rmse = np.sqrt(mean_squared_error(values, values_pred))
                print(f"RMSE bulan {month}: {rmse:.4f}")
            except Exception as e:
                print(f"Gagal MLP bulan {month}: {e}")
                rmse = np.nan
                continue

            rmse_records.append({
                'file': csv_file,
                'year': year,
                'month': month,
                'hidden_layers': str((150, 75, 30)),
                'rmse': rmse,
            })

            # --- Masking grid output dengan PNG/PGW ---
            xpix, ypix = lonlat_to_pixel(gridx_proj, gridy_proj, A, E, C, F)
            xi = np.round(xpix).astype(int)
            yi = np.round(ypix).astype(int)
            mask = (0 <= xi) & (xi < mask_land.shape[1]) & (0 <= yi) & (yi < mask_land.shape[0])
            mask_grid = np.zeros_like(z, dtype=bool)
            mask_grid[mask] = mask_land[yi[mask], xi[mask]]
            z_masked = np.ma.masked_where(~mask_grid, z)

            # --- Titik threshold 95% di darat untuk plot dan simpan file ---
            mask_95 = (values > perc95_rf) & on_land
            x_thresh, y_thresh = x[mask_95], y[mask_95]
            val_thresh = values[mask_95]
            x_thresh_pix, y_thresh_pix = lonlat_to_pixel(x_thresh, y_thresh, A, E, C, F)
            valid_titik = []
            for lon0, lat0, xp, yp, val0 in zip(x_thresh, y_thresh, x_thresh_pix, y_thresh_pix, val_thresh):
                dist = (gridx_proj - lon0)**2 + (gridy_proj - lat0)**2
                idx = np.unravel_index(np.argmin(dist), dist.shape)
                val_here = z_masked[idx]
                if mask_grid[idx] and (not np.ma.is_masked(val_here)) and (not np.isnan(val_here)) and (val_here > 0):
                    xi, yi = int(round(xp)), int(round(yp))
                    if (0 <= xi < mask_land.shape[1]) and (0 <= yi < mask_land.shape[0]):
                        valid_titik.append((xp, yp))
            valid_titik = np.array(valid_titik)

            # --- Simpan file titik threshold 95% dan nilainya ---
            if len(x_thresh) > 0:
                thres95_df = pd.DataFrame({
                    'csv_file': csv_file,
                    'year': year,
                    'month': month,
                    'lon': x_thresh,
                    'lat': y_thresh,
                    'rain': val_thresh
                })
                threshold_points_records.append(thres95_df)
                thres95_path = os.path.join(
                    output_dir,
                    f'{os.path.splitext(csv_file)[0]}_{year}_{month}_thres95_points.csv'
                )
                thres95_df.to_csv(thres95_path, index=False)
                print(f"Titik threshold 95% bulan {year}-{month} disimpan di {thres95_path}")

            # --- Simpan NetCDF ---
            try:
                basename = os.path.splitext(csv_file)[0]
                nc_path = os.path.join(output_dir, f'{basename}_{year}_{month}_mlp_150_75_30.nc')
                ds = xr.Dataset(
                    {
                        "tp": (("latitude", "longitude"), z.reshape(gridy_proj.shape)),
                        "is_land": (("latitude", "longitude"), mask_grid)
                    },
                    coords={
                        "longitude": gridx,
                        "latitude": gridy,
                        "year": year,
                        "month": month,
                        "datetime": datetime_value
                    }
                )
                ds.to_netcdf(nc_path)
                print(f"Data bulan {year}-{month} disimpan ke NetCDF di {nc_path}")
            except Exception as e:
                print(f"Gagal menyimpan NetCDF bulan {year}-{month}: {e}")

            # --- Plot dengan marking threshold 95% ---
            try:
                fig, ax = plt.subplots(figsize=(16, 6))
                ax.imshow(img_array, origin='upper')
                rain = ax.pcolormesh(xpix, ypix, z_masked, shading='auto', alpha=0.5, cmap='viridis', vmin=0, vmax=800)
                if len(valid_titik) > 0:
                    ax.scatter(valid_titik[:,0], valid_titik[:,1], color='white', s=30, marker='s', zorder=5)
                ax.set_xlim(0, img_array.shape[1])
                ax.set_ylim(img_array.shape[0], 0)
                plt.title(f'MLP {year}-{month} (Thres95={perc95_rf:.1f} mm)')
                plt.tight_layout()
                png_path = os.path.join(output_dir, f'{os.path.splitext(csv_file)[0]}_{year}_{month}_thres95_layeronly.png')
                plt.savefig(png_path, dpi=300, bbox_inches='tight')
                plt.close()
                print(f"Plot {year}-{month} disimpan di {png_path}")
            except Exception as e:
                print(f"Gagal membuat plot bulan {month}: {e}")

            # --- Evaluasi KFold di darat untuk threshold 95% ---
            try:
                pred_obs, fold_idx = mlp_kfold_eval(x_land, y_land, values_land, n_splits=5)
                obs = values_land
                pred = pred_obs
                threshold_eval = perc95_rf

                pod, far, csi, hits, misses, false_alarms, correct_negatives = compute_pod_far_csi(
                    obs, pred, threshold_eval
                )
                eval_dict = {
                    'csv_file': csv_file,
                    'year': year,
                    'month': month,
                    'thres95': threshold_eval,
                    'POD': pod,
                    'FAR': far,
                    'CSI': csi,
                    'hits': hits,
                    'misses': misses,
                    'false_alarms': false_alarms,
                    'correct_negatives': correct_negatives,
                    'n_obs': len(obs)
                }
                eval_list.append(eval_dict)
            except Exception as e:
                print(f"Gagal evaluasi bulan {year}-{month}: {e}")

    # --- Simpan hasil evaluasi, RMSE, dan threshold points ke CSV summary ---
    if rmse_records:
        rmse_df = pd.DataFrame(rmse_records)
        rmse_df.to_csv(os.path.join(output_dir, "mlp_150_75_30_rmse_per_bulan.csv"), index=False)
        print(f"File RMSE per bulan disimpan di {os.path.join(output_dir, 'mlp_150_75_30_rmse_per_bulan.csv')}")

    if eval_list:
        eval_df = pd.DataFrame(eval_list)
        eval_csv_path = os.path.join(output_dir, 'hasil_POD_FAR_CSI_bulanan.csv')
        eval_df.to_csv(eval_csv_path, index=False)
        print(f"Rekap evaluasi disimpan di {eval_csv_path}")

    if threshold_points_records:
        thres_all_df = pd.concat(threshold_points_records, ignore_index=True)
        thres_all_path = os.path.join(output_dir, 'all_bulan_thres95_points.csv')
        thres_all_df.to_csv(thres_all_path, index=False)
        print(f"File gabungan titik threshold 95% bulanan disimpan di {thres_all_path}")

    print(f"\nProses selesai! Semua data NetCDF, plot, dan rekap disimpan di {output_dir}")

if __name__ == "__main__":
    main()

## 8. Hasil & Visualisasi
Contoh keluaran akhir: rata-rata curah hujan musiman hasil pemodelan, dioverlay di atas peta topografi Pulau Jawa dan dibatasi hanya pada area daratan.

In [ ]:
#MUSIMAN
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from PIL import Image

# Fungsi baca file .pgw, hasil: [A, D, B, E, C, F]
def read_pgw(pgw_path):
    with open(pgw_path, 'r') as f:
        lines = [float(line.strip()) for line in f.readlines()]
    return lines  # [A, D, B, E, C, F]

# Fungsi lon/lat ke pixel
def lonlat_to_pixel(lon, lat, A, E, C, F):
    x_pix = (lon - C) / A
    y_pix = (lat - F) / E
    return x_pix, y_pix

# Path folder data dan topografi
input_dir = '/content/drive/MyDrive/IDW(PERSIANN)/topo'
topo_png_path = '/content/drive/MyDrive/JAWAAAA/JAWA.png'
topo_pgw_path = '/content/drive/MyDrive/JAWAAAA/JAWA.pgw'

# Baca PGW
A, D, B, E, C, F = read_pgw(topo_pgw_path)

# Baca gambar topografi dan buat mask daratan
img = Image.open(topo_png_path)
img_array = np.asarray(img)
if img_array.ndim == 3:
    gray = img_array.mean(axis=2)
else:
    gray = img_array
mask_land = gray < 250  # threshold sesuai workflow MLP

# Definisi musim
season_months = {
    'Basah': [12, 1, 2],
    'Transisi1': [3, 4, 5],
    'Kering': [6, 7, 8],
    'Transisi2': [9, 10, 11]
}

seasonal_sum = {k: None for k in season_months}
seasonal_count = {k: 0 for k in season_months}
lat, lon = None, None

# Loop file nc
for fname in sorted(os.listdir(input_dir)):
    if not fname.endswith('.nc'):
        continue
    fpath = os.path.join(input_dir, fname)
    ds = xr.open_dataset(fpath)
    # Ambil bulan
    if 'month' in ds.coords:
        month = int(ds['month'].values)
    elif 'month' in ds.attrs:
        month = int(ds.attrs['month'])
    else:
        parts = fname.split('_')
        month = int(parts[-2]) if parts[-2].isdigit() else None
    musim = None
    for s, bulan in season_months.items():
        if month in bulan:
            musim = s
            break
    if musim is None: continue

    tp = ds['tp'].values
    if seasonal_sum[musim] is None:
        seasonal_sum[musim] = np.zeros_like(tp, dtype=float)
    seasonal_sum[musim] += tp
    seasonal_count[musim] += 1
    if lat is None or lon is None:
        lat = ds['latitude'].values
        lon = ds['longitude'].values
    ds.close()

# Rata-rata musiman
seasonal_mean = {k: v / seasonal_count[k] if v is not None and seasonal_count[k] > 0 else None for k, v in seasonal_sum.items()}

# Bikin meshgrid lon/lat dan konversi ke pixel
lon2d, lat2d = np.meshgrid(lon, lat)
x_pix, y_pix = lonlat_to_pixel(lon2d, lat2d, A, E, C, F)
xi = np.round(x_pix).astype(int)
yi = np.round(y_pix).astype(int)

# Buat mask grid berdasarkan topografi PNG
mask_grid = np.zeros_like(lon2d, dtype=bool)
valid = (0 <= xi) & (xi < mask_land.shape[1]) & (0 <= yi) & (yi < mask_land.shape[0])
mask_grid[valid] = mask_land[yi[valid], xi[valid]]

# Plot musiman hanya di daratan
for musim, arr in seasonal_mean.items():
    if arr is None:
        print(f"Tidak ada data untuk musim {musim}")
        continue
    arr_masked = np.ma.masked_where(~mask_grid, arr)
    plt.figure(figsize=(14,7))
    plt.imshow(img_array, origin='upper')
    pm = plt.pcolormesh(
        x_pix, y_pix, arr_masked, shading='auto', cmap='viridis', alpha=0.7,
        vmin=0, vmax=500
    )
    plt.colorbar(pm, label='Rata-rata Hujan Musiman (mm)')
    plt.title(f"Musim {musim} - Rata-rata Hujan")
    plt.xlim(0, img_array.shape[1])
    plt.ylim(img_array.shape[0], 0)
    plt.xlabel('Pixel X')
    plt.ylabel('Pixel Y')
    plt.tight_layout()
    plt.savefig(f"{input_dir}/plot_musiman_pixel_{musim}_daratan_vmin0vmax500.png", dpi=300)
    plt.close()
    print(f"Plot musim {musim} (pixel, daratan, vmin0 vmax500) disimpan.")

print("Semua plot musiman (pixel, daratan, vmin0 vmax500) selesai.")

## 9. Evaluasi Akhir (Angka Final dari Laporan Skripsi)

Ketiga model dievaluasi terhadap data referensi IDW dari Yanto et al. (2017) menggunakan **MAE** (bulanan, musiman, tahunan) dan terhadap kejadian ekstrem menggunakan **POD/FAR/CSI**.

### 9.1 MAE Rata-Rata Tahunan (2001–2014)

| Model | MAE Rata-Rata Tahunan | MAE Rata-Rata Bulanan |
|---|---|---|
| **MLP** | **46,01** (terbaik) | **45,15** (terbaik) |
| IDW2 | 47,44 | 48,37 |
| IDW1 | 49,01 | 49,43 |

### 9.2 MAE Musiman

| Musim | MAE MLP | MAE IDW1 | MAE IDW2 |
|---|---|---|---|
| Basah (Des–Feb) | **67,25** | 75,04 | 74,06 |
| Transisi 1 (Mar–Mei) | **53,82** | 60,29 | 59,11 |
| Kering (Jun–Agu) | **20,89** | 22,24 | 21,37 |
| Transisi 2 (Sep–Nov) | **45,93** | 49,16 | 47,58 |

### 9.3 Evaluasi Curah Hujan Ekstrem (persentil ≥95%)

| Model | POD | FAR | CSI |
|---|---|---|---|
| **IDW2** | **0,619** (terbaik) | **0,117** (terkecil) | **0,579** (terbaik) |
| IDW1 | 0,472 | 0,167 | 0,419 |
| MLP | 0,151 | 0,638 | 0,118 |

### Kesimpulan
- **MLP** adalah model dengan performa **terbaik secara umum** untuk estimasi curah hujan bulanan, musiman, dan tahunan (MAE terendah di semua rentang waktu).
- **IDW2**, meskipun MAE-nya sedikit lebih tinggi dari MLP, justru **jauh lebih unggul dalam mendeteksi curah hujan ekstrem** (CSI 0,579 vs MLP hanya 0,118) — MLP cenderung melewatkan (miss) kejadian ekstrem, sedangkan IDW2 jauh lebih sensitif terhadapnya.
- Implikasinya: MLP cocok untuk estimasi curah hujan rata-rata secara umum, sementara IDW2 lebih andal untuk aplikasi yang butuh deteksi kejadian ekstrem (mis. peringatan dini banjir).
- Saran pengembangan (dari skripsi): menambahkan variabel eksogen (VIMFC, OLR, dll.) untuk MLP, dan mengeksplorasi pendekatan **hybrid IDW2 + MLP** — MLP untuk nilai umum, IDW2 untuk nilai ekstrem.